# Multi-Encoder U-Net – Brain Tumor Segmentation (BraTS)

Clean main notebook ready for GitHub.

In [ ]:

# Imports
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from tensorflow.keras.layers import Input
from tensorflow import keras

from datagenerator import DataGenerator
from unet_model import multi_encoder_unet
from metrics import (
    dice_coef, dice_coef_necrotic, dice_coef_edema,
    dice_coef_enhancing, dice_tumor_core, dice_whole_tumor
)


In [ ]:

# Paths & Params
TRAIN_DATASET_PATH = './BraTS2021_Training_Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData'
VALIDATION_DATASET_PATH = './BraTS2021_Validation_Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-ValidationData'
IMG_SIZE = 192


In [ ]:

def extract_ids(path):
    return [f.name for f in os.scandir(path) if f.is_dir()]

train_ids = extract_ids(TRAIN_DATASET_PATH)
val_ids = extract_ids(VALIDATION_DATASET_PATH)

print(len(train_ids), "training cases")
print(len(val_ids), "validation cases")


In [ ]:

# Model
input1 = Input((IMG_SIZE, IMG_SIZE, 2))
input2 = Input((IMG_SIZE, IMG_SIZE, 1))
input3 = Input((IMG_SIZE, IMG_SIZE, 1))

model = multi_encoder_unet(input1, input2, input3)
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=[
        'accuracy', dice_coef, dice_coef_necrotic,
        dice_coef_edema, dice_coef_enhancing,
        dice_tumor_core, dice_whole_tumor
    ]
)
model.summary()


In [ ]:

# Load pretrained model (optional)
MODEL_PATH = "./modelC.h5"
if os.path.exists(MODEL_PATH):
    model = keras.models.load_model(
        MODEL_PATH,
        custom_objects={
            "dice_coef": dice_coef,
            "dice_coef_necrotic": dice_coef_necrotic,
            "dice_coef_edema": dice_coef_edema,
            "dice_coef_enhancing": dice_coef_enhancing,
            "dice_tumor_core": dice_tumor_core,
            "dice_whole_tumor": dice_whole_tumor
        },
        compile=False
    )
    print("Model loaded")


In [ ]:

# Inference
def seg_case(case, slice_=50, data_type="validation"):
    base = TRAIN_DATASET_PATH if data_type=="training" else VALIDATION_DATASET_PATH
    cp = os.path.join(base, case)

    flair = nib.load(os.path.join(cp, f"{case}-t2f.nii.gz")).get_fdata()
    ce = nib.load(os.path.join(cp, f"{case}-t1c.nii.gz")).get_fdata()
    t1 = nib.load(os.path.join(cp, f"{case}-t1n.nii.gz")).get_fdata()
    t2 = nib.load(os.path.join(cp, f"{case}-t2w.nii.gz")).get_fdata()

    X = np.zeros((155, IMG_SIZE, IMG_SIZE, 2))
    Z = np.zeros((155, IMG_SIZE, IMG_SIZE, 1))
    W = np.zeros((155, IMG_SIZE, IMG_SIZE, 1))

    for i in range(155):
        X[i,:,:,0] = cv2.resize(flair[:,:,i], (IMG_SIZE, IMG_SIZE))
        X[i,:,:,1] = cv2.resize(ce[:,:,i], (IMG_SIZE, IMG_SIZE))
        Z[i,:,:,0] = cv2.resize(t2[:,:,i], (IMG_SIZE, IMG_SIZE))
        W[i,:,:,0] = cv2.resize(t1[:,:,i], (IMG_SIZE, IMG_SIZE))

    p = model.predict([(X-X.mean())/X.std(), (Z-Z.mean())/Z.std(), (W-W.mean())/W.std()], verbose=0)
    pred = np.argmax(p, axis=3)

    fig, ax = plt.subplots(1,5, figsize=(20,5))
    ax[0].imshow(flair[:,:,slice_], cmap='gray')
    ax[1].imshow(t1[:,:,slice_], cmap='gray')
    ax[2].imshow(ce[:,:,slice_], cmap='gray')
    ax[3].imshow(t2[:,:,slice_], cmap='gray')
    ax[4].imshow(pred[slice_])
    for a in ax: a.axis('off')
    plt.show()


In [ ]:

# Run validation examples
for cid in val_ids[:3]:
    seg_case(cid)
